In [1]:
## Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import time
import random

In [3]:
## Webscrapping Request Generation check

In [4]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9"
}

response=requests.get('https://www.ambitionbox.com/list-of-companies?campaign=desktop_nav&page=1',headers=headers).text


In [5]:
## Intializing variable

In [6]:
soup=BeautifulSoup(response,'lxml')


In [7]:
## Extracting name of company from page 1

In [8]:
for i in soup.find_all('h2')[1:]:
    print(i.text.strip())

TCS
Accenture
Wipro
Cognizant
Capgemini
HDFC Bank
Infosys
HCLTech
ICICI Bank
Tech Mahindra
Genpact
TP
Axis Bank
Jio
Concentrix Corporation
Amazon
Reliance Retail
iEnergizer
LTM Limited
HDB Financial Services
Popular Collections by Industries
Popular Collections by Cities
Popular Collections by Roles


In [9]:
## Extracting rating of company from page 1

In [10]:
for i in soup.find_all('div',class_='rating_text'):
    print(i.text.strip())

3.3
3.7
3.6
3.7
3.6
3.8
3.5
3.4
3.9
3.3
3.6
3.9
3.6
4.4
3.5
3.9
3.9
4.6
3.6
3.9


In [11]:
## Extracting rating of company from page 1

In [12]:
for i in soup.find_all('span',class_='companyCardWrapper__companyRatingCount'):
    print(i.text.strip().replace('(','').replace(')',''))

1.2L
75.3k
66.5k
62.7k
54.8k
53.9k
50k
47.1k
46.8k
44.3k
43.5k
39.7k
34.2k
34.1k
33k
32.4k
27.8k
27.5k
27k
26.3k


In [13]:
## Extracting types of company from page 1

In [14]:
j=[]
for i in soup.find_all('span',class_='companyCardWrapper__interLinking'):
    j.append(i.text.strip().split('|')[0])
j


['IT Services & Consulting ',
 'IT Services & Consulting ',
 'IT Services & Consulting ',
 'IT Services & Consulting ',
 'IT Services & Consulting ',
 'Banking ',
 'IT Services & Consulting ',
 'IT Services & Consulting ',
 'Banking ',
 'IT Services & Consulting ',
 'Analytics & KPO ',
 'BPO ',
 'Banking ',
 'Telecom ',
 'BPO ',
 'Internet ',
 'Retail ',
 'BPO ',
 'IT Services & Consulting ',
 'NBFC ']

In [15]:
## final code: Extracting name, rating, count of votes,types of comapny, hq, salary, jobs count, benefits 
## from all available webpages

In [4]:
final=pd.DataFrame()
dfs=[]
session=requests.Session()
for a in range(1,500):
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 Safari/537.36",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9"
    }
    
    response=session.get(f'https://www.ambitionbox.com/list-of-companies?campaign=desktop_nav&page={a}',headers=headers,timeout=10)
    print('page',a,':',response.status_code)
    if response.status_code==403:
        print('Blocked by Ambition Box')
        break
    if response.status_code!=200:
        continue
    soup=BeautifulSoup(response.text,'lxml')
    
    name=[]
    rating=[]
    count=[]
    info=[]
    types=[]
    hq=[]
    salary=[]
    interview=[]
    jobs=[]
    benefits=[]
    company=soup.find_all('div',class_='companyCardWrapper')
    print('Page:',a,'-',len(company),'companies')
    if not company:
            break
    for i in company:
        try:
            name.append(i.find('h2').text.strip())
        except:
            name.append(np.nan)
        try:
            rating.append(i.find('div',class_='rating_text').text.strip())
        except:
            rating.append(np.nan)
        try:
            count.append(i.find('span',class_='companyCardWrapper__companyRatingCount').text.strip().replace('(','').replace(')',''))
        except:
            count.append(np.nan)
        
        try:
            j=i.find('span',class_='companyCardWrapper__interLinking').text.strip()
        except:
            j=''
        info.append(j)
        
        if '|' in j:
            x=j.split('|') 
            k=x[0].strip() 
            l=x[1].strip()  
            m=l.split('+')[0].strip()
        else:
            x=j.split('+')
            k=np.nan
            l=np.nan
            m=x[0].strip() if j else np.nan
            
        try:
            types.append(k)
        except:
            types.append(np.nan)
        try:
            hq.append(m)
        except:
            hq.append(np.nan)
        action=i.find_all('span',class_='companyCardWrapper__ActionCount')
        try:
            salary.append(action[1].text.strip())
        except:
            salary.append(np.nan)
       
        try:
            interview.append(action[2].text.strip())
        except:
            interview.append(np.nan)
        try:
            jobs.append(action[3].text.strip())
        except:
            jobs.append(np.nan)
        try:
            benefits.append(action[4].text.strip())
        except:
            benefits.append(np.nan)

    temp_df=pd.DataFrame({
        'name':name,
        'ratings':rating,
        'count_of_rating':count,
        'company_types':types,
        'hq':hq,
        'salary':salary,
        'interview_counts':interview,
        'jobs':jobs,
        'benefits':benefits
    })
    dfs.append(temp_df)
    time.sleep(random.uniform(2,5))
final=pd.concat(dfs,ignore_index=True)
final
    

page 1 : 200
Page: 1 - 20 companies
page 2 : 200
Page: 2 - 20 companies
page 3 : 200
Page: 3 - 20 companies
page 4 : 200
Page: 4 - 20 companies
page 5 : 200
Page: 5 - 20 companies
page 6 : 200
Page: 6 - 20 companies
page 7 : 200
Page: 7 - 20 companies
page 8 : 200
Page: 8 - 20 companies
page 9 : 200
Page: 9 - 20 companies
page 10 : 200
Page: 10 - 20 companies
page 11 : 200
Page: 11 - 20 companies
page 12 : 200
Page: 12 - 20 companies
page 13 : 200
Page: 13 - 20 companies
page 14 : 200
Page: 14 - 20 companies
page 15 : 200
Page: 15 - 20 companies
page 16 : 200
Page: 16 - 20 companies
page 17 : 200
Page: 17 - 20 companies
page 18 : 200
Page: 18 - 20 companies
page 19 : 200
Page: 19 - 20 companies
page 20 : 200
Page: 20 - 20 companies
page 21 : 200
Page: 21 - 20 companies
page 22 : 200
Page: 22 - 20 companies
page 23 : 200
Page: 23 - 20 companies
page 24 : 200
Page: 24 - 20 companies
page 25 : 200
Page: 25 - 20 companies
page 26 : 200
Page: 26 - 20 companies
page 27 : 200
Page: 27 - 20 co

,name,ratings,count_of_rating,company_types,hq,salary,interview_counts,jobs,benefits
0,TCS,3.3,1.2L,IT Services & Consulting,Bengaluru,10.4L,11.4k,5.1k,10.9k
1,Accenture,3.7,75.6k,IT Services & Consulting,Bengaluru,7.2L,9.5k,19.7k,6.9k
2,Wipro,3.6,66.7k,IT Services & Consulting,Hyderabad,4.9L,7k,182,4.9k
3,Cognizant,3.7,63k,IT Services & Consulting,Hyderabad,6.1L,6.6k,906,5.6k
4,Capgemini,3.6,55.1k,IT Services & Consulting,Bengaluru,5L,5.7k,2.1k,3.8k
...,...,...,...,...,...,...,...,...,...
9975,Rayat Bahra University,4.5,115,Education & Training,Chandigarh,166,8,--,11
9976,Delta Exchange,4.8,115,Financial Services,Bengaluru,202,37,3,1
9977,East West Seeds,4.4,115,FMCG,Chhatrapati Sambhajinagar,394,10,--,11
9978,SPS Construction & Engineering Works,3.3,115,Engineering & Construction,Chennai,544,6,--,8


In [9]:
final.to_csv('Ambitionbox_Companies_Raw.csv',index=False)

In [17]:
# Understanding The Data

In [10]:
final.sample(5)

,name,ratings,count_of_rating,company_types,hq,salary,interview_counts,jobs,benefits
663,Wockhardt,3.8,1.6k,Pharma,Chhatrapati Sambhajinagar,4.2k,67,29,183
7385,CLIRNet,3.6,158,Internet,Kolkata,509,14,24,6
732,Hyatt Regency,4.0,1.5k,Hospitality,New Delhi,3.1k,64,1,198
8155,Dinesh Engineers,3.9,142,Telecom,Mumbai,501,12,1,8
1397,Alivus Life Sciences,3.8,792,Pharma,Ankleshwar,4.1k,42,--,46


In [11]:
final.shape

(9980, 9)

In [12]:
final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9980 entries, 0 to 9979
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   name              9980 non-null   object
 1   ratings           9971 non-null   object
 2   count_of_rating   9971 non-null   object
 3   company_types     9696 non-null   object
 4   hq                9980 non-null   object
 5   salary            9980 non-null   object
 6   interview_counts  9980 non-null   object
 7   jobs              9980 non-null   object
 8   benefits          9980 non-null   object
dtypes: object(9)
memory usage: 701.8+ KB


In [13]:
final.describe()

,name,ratings,count_of_rating,company_types,hq,salary,interview_counts,jobs,benefits
count,9980,9971,9971,9696,9980,9980,9980,9980,9980
unique,9559,38,945,83,257,1232,470,319,456
top,Gammastack,3.9,1.1k,IT Services & Consulting,Mumbai,1.1k,7,--,12
freq,2,1030,96,1312,1537,401,382,3586,367


In [22]:
# Checking & Removing Dulpicates

In [14]:
final[final.duplicated()]

,name,ratings,count_of_rating,company_types,hq,salary,interview_counts,jobs,benefits
180,State Street Corporation,3.5,4.3k,Financial Services,Bengaluru,35k,305,10,274
320,SRF,4.1,2.9k,Chemicals,Bharuch,9.4k,215,19,260
680,GE VERNOVA,4.1,1.6k,Power,Bengaluru,5.8k,115,6,14
860,Prodapt,3.5,1.3k,IT Services & Consulting,Chennai,10k,120,93,77
1320,Y-Axis Overseas Careers,4.1,839,Other,Hyderabad,1.5k,158,3,241
...,...,...,...,...,...,...,...,...,...
9965,Teqfocus Consulting,3.8,115,IT Services & Consulting,Bengaluru,677,11,4,1
9966,Arora Iron & Steel Rolling Mills,3.8,115,Iron & Steel,Ludhiana,481,7,--,5
9967,Brennan,3.7,115,IT Services & Consulting,Mumbai,545,3,6,7
9968,ekincare,3.0,115,Healthcare,Hyderabad,617,12,1,7


In [15]:
final=final.drop_duplicates().reset_index(drop=True)

In [16]:
final[final.duplicated()]

,name,ratings,count_of_rating,company_types,hq,salary,interview_counts,jobs,benefits


In [26]:
# Checking Null Values 

In [17]:
final.isnull().sum()

name                  0
ratings               9
count_of_rating       9
company_types       268
hq                    0
salary                0
interview_counts      0
jobs                  0
benefits              0
dtype: int64

In [ ]:
# Formatting the data

In [18]:
def convert_k(x):
    if pd.isna(x):
        return np.nan
    x=str(x).lower().replace(',','')
    if x.endswith('k'):
        return float(str(x).replace('k',''))*1000
    elif x.endswith('l'):
        return float(str(x).replace('l',''))*100000
    elif x=='--':
        return np.nan
        
    return float(x)

cols=['ratings','count_of_rating','salary','interview_counts','jobs','benefits']

for col in cols:
    final[col]=final[col].apply(convert_k)

In [28]:
# Replacing Null Values

In [19]:
for col in cols:
    
    final[col]=final[col].fillna(0)


In [20]:
final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9563 entries, 0 to 9562
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   name              9563 non-null   object 
 1   ratings           9563 non-null   float64
 2   count_of_rating   9563 non-null   float64
 3   company_types     9295 non-null   object 
 4   hq                9563 non-null   object 
 5   salary            9563 non-null   float64
 6   interview_counts  9563 non-null   float64
 7   jobs              9563 non-null   float64
 8   benefits          9563 non-null   float64
dtypes: float64(6), object(3)
memory usage: 672.5+ KB


In [ ]:
# Checking null value after null handling

In [21]:
final.isnull().sum()

name                  0
ratings               0
count_of_rating       0
company_types       268
hq                    0
salary                0
interview_counts      0
jobs                  0
benefits              0
dtype: int64

In [23]:
final.loc[1]

name                               Accenture
ratings                                  3.7
count_of_rating                      75600.0
company_types       IT Services & Consulting
hq                                 Bengaluru
salary                              720000.0
interview_counts                      9500.0
jobs                                 19700.0
benefits                              6900.0
Name: 1, dtype: object

In [ ]:
# Final Clean Data before Analysis

In [24]:
final.to_csv('ambitionbox_companies_cleaned.csv',index=False)


In [ ]:
# checking where file is saved

In [25]:
import os
os.getcwd()

'C:\\Users\\soumy\\My Folder'